In [1]:
import pandas as pd
import numpy as np
import os
import sys
%load_ext autoreload
%autoreload 2
# sys.path.append('/home/wolfgang/repos/sleep_research_io')
# from sleep_research_functions import *
%matplotlib widget
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import seaborn as sns
from icu_sleep_breathing_2020_help_functions import * 

pd.options.display.max_rows = 300
pd.options.display.max_columns = 300

font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 8}

font_headers =  {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 9}

font_subplots =  {'family' : 'sans-serif',
        'weight' : 'bold',
        'size'   : 9}

matplotlib.rc('font', **font)


In [2]:

def default_scatter():
    plt.figure(figsize=figsize)
    plt.scatter(Xy_sleeplab[x_var], Xy_sleeplab[y_var], c='goldenrod', s=s, alpha=alpha) # orange
    plt.scatter(Xy_icu[x_var], Xy_icu[y_var], c='navy', s=s, alpha=alpha-0.1)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend(['Sleeplab', 'ICU'])
    plt.title(title)
    plt.tight_layout()
    
    
def default_scatter_ax0(ax):
    ax.scatter(Xy_sleeplab[x_var], Xy_sleeplab[y_var], c='goldenrod', s=s, alpha=alpha)
    ax.scatter(Xy_icu[x_var], Xy_icu[y_var], c='navy', s=s, alpha=alpha-0.1)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend(['Sleeplab', 'ICU'])
    ax.set_title(title)
    
    
def default_scatter_ax(ax, x_var, y_var, title):
    ax.scatter(Xy_sleeplab[x_var], Xy_sleeplab[y_var], c='goldenrod', s=s, alpha=alpha)
    ax.scatter(Xy_icu[x_var], Xy_icu[y_var], c='navy', s=s, alpha=alpha-0.1)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
#     ax.legend(['Sleeplab', 'ICU'])
    ax.set_title(title)
    
    
def barplot_annotate_brackets(num1, num2, data, center, height, ax, yerr=None, dh=.05, barh=.01, fs=None, maxasterix=None):
    """ 
    Annotate barplot with p-values.

    :param num1: number of left bar to put bracket over
    :param num2: number of right bar to put bracket over
    :param data: string to write or number for generating asterixes
    :param center: centers of all bars (like plt.bar() input)
    :param height: heights of all bars (like plt.bar() input)
    :param yerr: yerrs of all bars (like plt.bar() input)
    :param dh: height offset over bar / bar + yerr in axes coordinates (0 to 1)
    :param barh: bar height in axes coordinates (0 to 1)
    :param fs: font size
    :param maxasterix: maximum number of asterixes to write (for very small p-values)
    """

    if type(data) is str:
        text = data
    else:
        # * is p < 0.05
        # ** is p < 0.005
        # *** is p < 0.0005
        # etc.
        text = ''
        p = .05

        while data < p:
            text += '*'
            p /= 10.

            if maxasterix and len(text) == maxasterix:
                break

        if len(text) == 0:
            text = 'n. s.'

    lx, ly = center[num1], height[num1]
    rx, ry = center[num2], height[num2]

    if yerr:
        ly += yerr[num1]
        ry += yerr[num2]

    ax_y0, ax_y1 = ax.get_ylim()
    dh *= (ax_y1 - ax_y0)
    barh *= (ax_y1 - ax_y0)

    y = max(ly, ry) + dh

    barx = [lx, lx, rx, rx]
    bary = [y, y+barh, y+barh, y]
    mid = ((lx+rx)/2, y+barh)

#     ax.plot(barx, bary, c='black', lw=0.5)

    kwargs = dict(ha='center', va='bottom')
    if fs is not None:
        kwargs['fontsize'] = fs

#     ax.text(*mid, text, **kwargs)
    
#     np.max(height)
    y = ax_y1 *0.86 - 0.005 * int(ax_y1<1)
    ax.text(*(rx, y), text, **kwargs)


In [3]:
ahi_categories = ['all', 'ahi_0_5', 'ahi_5_15', 'ahi_15_30', 'ahi_30_100']

ahi_names = {
    'all': 'All',
    'ahi_0_5': 'AHI < 5',
    'ahi_5_15': '5 < AHI <=15',
    'ahi_15_30': '15 < AHI <= 30',
    'ahi_30_100': 'AHI > 30',
}

breathing_vars = ['rr_10s_smooth', 'instability_30sec', 'ibi', 'ventilation_10s_smooth', 'ventilation_cvar_30sec', 'inht_cycle_ratio_10sec']

stages = ['W', 'S', 'R', 'N1', 'N2', 'N3']
stages_titles = ['Wake', 'Sleep', 'R', 'N1', 'N2', 'N3']

### load summary tables:

In [4]:
plots_savedir = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/plots'
plots_savedir = '/scr/wolfgang/Dropbox (Partners HealthCare)/SleepInICUandSleeplabPaper/Plots/breathing'
tables_savedir = '/scr/wolfgang/Dropbox (Partners HealthCare)/SleepInICUandSleeplabPaper/Results'


In [5]:
[summary_subjects_icu, summary_subjects_sleeplab, 
 summary_days_icu, summary_days_sleeplab] = load_summary_data_with_inclusion_criteria()

print(summary_days_icu.shape)
print(summary_days_sleeplab.shape)

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3254: DtypeWarning: Columns (15,91,92,93,6455,6456,6457,12819,12820,12821,19185,19186,19187,21491,21492,21493,21519,21520,21521,21526,21527,21528,25549,25550,25551,31913,31914,31915,38279,38280,38281,44643,44644,44645,46851,46852,46853,46865,46866,46867,46879,46880,46881,46949,46950,46951,46956,46957,46958,46963,46964,46965,46970,46971,46972,46977,46978,46979,46984,46985,46986,51007,51008,51009,53336,53337,53338,57305,57373,57374,57375,59588,59589,59590,59602,59603,59604,59686,59687,59688,63669,63737,63738,63739,65952,65953,65954,65959,65960,65961,65980,65981,65982,66036,66037,66038,66043,66044,66045,66050,66051,66052,66057,66058,66059,66071,66072,66073,66078,66079,66080,70033,70101,70102,70103,72318,72319,72320,72332,72333,72334,72402,72403,72404,72416,72417,72418,76399,76467,76468,76469,78682,78683,78684,78696,78697,78698,78710,78711,78712,78766,78767,78768,78787,78788,78789,78801,78

# of subjects ICU before inclusion criteria: 102
# of 12-hour segments ICU before inclusion criteria: 1161
# of subjects ICU after inclusion criteria: 102
# of 12-hour segments ICU after inclusion criteria: 1161
24-hour segments: 387
12-hour segments: 774

# of subjects sleeplab before inclusion criteria: 412
# of 12-hour segments sleeplab before inclusion criteria: 412
(1161, 6556)
(412, 6126)


In [6]:
for agreement in ['all', 'agreement_relaxed', 'disagreement_relaxed', 'agreement', 'disagreement']:
    for modality in ['amendsumeffort', 'ecg_nn']:
        var_n2 = 'perc_N2_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement)
        var_n3 = 'perc_N3_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement)
        var_sum = 'perc_N2N3_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement)

        summary_days_icu[var_sum] = summary_days_icu[var_n2] + summary_days_icu[var_n3]
        summary_days_sleeplab[var_sum] = summary_days_sleeplab[var_n2] + summary_days_sleeplab[var_n3]
        
        
for agreement in ['agreement_relaxed', 'agreement', 'all']:
    for modality in ['amendsumeffort', 'ecg_nn']:
        var_sleep = 'hours_sleep_MODALITY_all'.replace('MODALITY', modality) # all overlap sleep
        var_agreeing = 'hours_sleep_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement)
        
        var_discordant = 'hours_discordant_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement) # discordant sleep among overlap sleep
        summary_days_icu[var_discordant] = summary_days_icu[var_sleep] - summary_days_icu[var_agreeing]
        summary_days_sleeplab[var_discordant] = summary_days_sleeplab[var_sleep] - summary_days_sleeplab[var_agreeing]
        
        var_discordant_perc = 'perc_discordant_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement) # discordant sleep among overlap sleep in %
        summary_days_icu[var_discordant_perc] = np.divide(summary_days_icu[var_sleep] - summary_days_icu[var_agreeing], summary_days_icu[var_sleep])
        summary_days_sleeplab[var_discordant_perc] = np.divide(summary_days_sleeplab[var_sleep] - summary_days_sleeplab[var_agreeing], summary_days_sleeplab[var_sleep])
        
for var_tmp in breathing_vars:
    summary_days_icu[var_tmp + '_iqr'] = summary_days_icu[var_tmp + '_q75'] - summary_days_icu[var_tmp + '_q25']
    summary_days_sleeplab[var_tmp + '_iqr'] = summary_days_sleeplab[var_tmp + '_q75'] -  summary_days_sleeplab[var_tmp + '_q25']
    

In [7]:
cols_percentage = [x for x in summary_days_sleeplab.columns if 'perc_' in x]
summary_days_sleeplab[cols_percentage] *= 100
summary_days_icu[cols_percentage] *= 100

summary_f_icu = summary_days_icu.loc[summary_days_icu.day_cat == 'f', :]
summary_dn_icu = summary_days_icu.loc[summary_days_icu.day_cat != 'f', :]

In [8]:
### 3x2 plot
# cols: data type, rows: stage version (resp, ecg, expert)

variables_template = ['sleep_hours_MODALITY',
             'stages_distribution_MODALITY_S',
             'stages_distribution_MODALITY_R',
             'stages_distribution_MODALITY_N1',
            'stages_distribution_MODALITY_N2',
             'stages_distribution_MODALITY_N3',
             'sleep_MODALITY_sfi',
             'sleep_MODALITY_sfi_w',
             'sleep_MODALITY_arousali',
            ]
x_labels = ['Sleep Dur. (h)', 'Sleep Efficiency', 'Stage R (%)', 'Stage N1 (%)', 'Stage N2 (%)', 'Stage N3 (%)', 'SFI', 'SFI Wake', 'Arousal I.']

# select version:

In [9]:
## compare ICU group 2 and sleeplab group 2, compute breathing/hrv statistics on ALL sleep:
# agreement = 'all'
# agreement_short = 'all'
# min_hours_sleep_icu = 0.01
# inclusion_agreement = 'all'
# do_agreement_for_sleeplab = False

# ## compare ICU group 3 and sleeplab group 3, compute breathing/hrv statistics on ALL sleep:
# agreement = 'all'
# agreement_short = 'all'
# min_hours_sleep_icu = 2
# inclusion_agreement = 'agreement_relaxed'
# do_agreement_for_sleeplab = True

### !!!!!!!!!!!!!!!!!!! The following version is actually the only version possible right now as stagewise features are only computed for
### agreeing sleep. if other versions are needed, 'all' agreeing option with stagewise needs to be added to summary-extraction script 

## compare ICU group 3 and sleeplab group 3, compute breathing/hrv statistics on CONCORDANT sleep:
agreement = 'agreement_relaxed'
agreement_short = 'agrelaxed'
min_hours_sleep_icu = 2
inclusion_agreement = 'agreement_relaxed'
do_agreement_for_sleeplab = True

# ## compare ICU group 3 and sleeplab group 2: (NOT INCLUDED IN PAPER)
# agreement = 'all'
# min_hours_sleep_icu = 2
# inclusion_agreement = 'agreement_relaxed'
# do_agreement_for_sleeplab = False


icu_day_v = 'full'

#### MODE: either median/iqr or mean/std are used for plots and statistical tests.
# currently, MannWhitney U test is applied, so median is assumed, for mean, code might need to be changed (e.g. t tests)

mode = 'median'

if mode == 'median':
    x_var = 'median'
    y_var = 'iqr'
elif mode == 'mean':
    x_var = 'mean'
    y_var = 'std'

In [10]:
if icu_day_v == 'day_night':
    summary_dn_icu_sel = summary_dn_icu.loc[np.any([summary_dn_icu[f'hours_sleep_amendsumeffort_{inclusion_agreement}'] >= min_hours_sleep_icu, 
                                                   summary_dn_icu[f'hours_sleep_ecg_nn_{inclusion_agreement}'] >= min_hours_sleep_icu],
                                                   axis=0), :].copy()
elif icu_day_v == 'full':
    summary_dn_icu_sel = summary_f_icu.loc[np.any([summary_f_icu[f'hours_sleep_amendsumeffort_{inclusion_agreement}'] >= min_hours_sleep_icu, 
                                                   summary_f_icu[f'hours_sleep_ecg_nn_{inclusion_agreement}'] >= min_hours_sleep_icu],
                                                   axis=0), :].copy()
    
summary_dn_icu_sel['strata'] = 'ICU'
summary_dn_icu_sel['strata_int'] = 99

In [26]:
sleeplab_all.shape

(190, 6514)

### make separate objects for each sleeplab strata

In [11]:
sleeplab_all = summary_days_sleeplab.loc[summary_days_sleeplab['matched_all'] == 1].copy()
sleeplab_ahi_0_5 = summary_days_sleeplab.loc[summary_days_sleeplab['matched_ahi_0_5'] == 1].copy()
sleeplab_ahi_5_15 = summary_days_sleeplab.loc[summary_days_sleeplab['matched_ahi_5_15'] == 1].copy()
sleeplab_ahi_15_30 = summary_days_sleeplab.loc[summary_days_sleeplab['matched_ahi_15_30'] == 1].copy()
sleeplab_ahi_30_100 = summary_days_sleeplab.loc[summary_days_sleeplab['matched_ahi_30_100'] == 1].copy()
sleeplab_ahi_15_100 = summary_days_sleeplab.loc[summary_days_sleeplab['matched_ahi_15_100'] == 1].copy()

sleeplab_ahi_0_5['strata'] = 'ahi_0_5'
sleeplab_ahi_0_5['strata_int'] = 1
sleeplab_ahi_15_100['strata'] = 'ahi_15_100'
sleeplab_ahi_15_100['strata_int'] = 5

if do_agreement_for_sleeplab:

    sleeplab_all = sleeplab_all.loc[np.any([sleeplab_all[f'hours_sleep_amendsumeffort_{inclusion_agreement}'] >= min_hours_sleep_icu, 
                                           sleeplab_all[f'hours_sleep_ecg_nn_{inclusion_agreement}'] >= min_hours_sleep_icu],
                                           axis=0), :]
    
    sleeplab_ahi_0_5 = sleeplab_ahi_0_5.loc[np.any([sleeplab_ahi_0_5[f'hours_sleep_amendsumeffort_{inclusion_agreement}'] >= min_hours_sleep_icu, 
                                           sleeplab_ahi_0_5[f'hours_sleep_ecg_nn_{inclusion_agreement}'] >= min_hours_sleep_icu],
                                           axis=0), :]
    
    sleeplab_ahi_5_15 = sleeplab_ahi_5_15.loc[np.any([sleeplab_ahi_5_15[f'hours_sleep_amendsumeffort_{inclusion_agreement}'] >= min_hours_sleep_icu, 
                                           sleeplab_ahi_5_15[f'hours_sleep_ecg_nn_{inclusion_agreement}'] >= min_hours_sleep_icu],
                                           axis=0), :]
        
    sleeplab_ahi_15_30 = sleeplab_ahi_15_30.loc[np.any([sleeplab_ahi_15_30[f'hours_sleep_amendsumeffort_{inclusion_agreement}'] >= min_hours_sleep_icu, 
                                           sleeplab_ahi_15_30[f'hours_sleep_ecg_nn_{inclusion_agreement}'] >= min_hours_sleep_icu],
                                           axis=0), :]

    sleeplab_ahi_15_100 = sleeplab_ahi_15_100.loc[np.any([sleeplab_ahi_15_100[f'hours_sleep_amendsumeffort_{inclusion_agreement}'] >= min_hours_sleep_icu, 
                                           sleeplab_ahi_15_100[f'hours_sleep_ecg_nn_{inclusion_agreement}'] >= min_hours_sleep_icu],
                                           axis=0), :]
    
    sleeplab_ahi_30_100 = sleeplab_ahi_30_100.loc[np.any([sleeplab_ahi_30_100[f'hours_sleep_amendsumeffort_{inclusion_agreement}'] >= min_hours_sleep_icu, 
                                           sleeplab_ahi_30_100[f'hours_sleep_ecg_nn_{inclusion_agreement}'] >= min_hours_sleep_icu],
                                           axis=0), :]

In [12]:
print(f'Min Hours of Sleep: {min_hours_sleep_icu}')
print(f'{icu_day_v} Segments included:')
print(f'ICU:      {summary_dn_icu_sel.shape[0]} ({summary_dn_icu_sel.study_id.unique().shape[0]} subjects)')
print(f'Sleeplab AHI All: {sleeplab_all.shape[0]} ({sleeplab_all.study_id.unique().shape[0]} subjects)')
print(f'Sleeplab AHI [0,5]: {sleeplab_ahi_0_5.shape[0]} ({sleeplab_ahi_0_5.study_id.unique().shape[0]} subjects)')
print(f'Sleeplab AHI [5,15]: {sleeplab_ahi_5_15.shape[0]} ({sleeplab_ahi_5_15.study_id.unique().shape[0]} subjects)')
print(f'Sleeplab AHI [15, 30]: {sleeplab_ahi_15_30.shape[0]} ({sleeplab_ahi_15_30.study_id.unique().shape[0]} subjects)')
print(f'Sleeplab AHI [30, 100]: {sleeplab_ahi_30_100.shape[0]} ({sleeplab_ahi_30_100.study_id.unique().shape[0]} subjects)')
print(f'Sleeplab AHI [15, 100]: {sleeplab_ahi_15_100.shape[0]} ({sleeplab_ahi_15_100.study_id.unique().shape[0]} subjects)')

Min Hours of Sleep: 2
full Segments included:
ICU:      163 (80 subjects)
Sleeplab AHI All: 190 (190 subjects)
Sleeplab AHI [0,5]: 69 (69 subjects)
Sleeplab AHI [5,15]: 60 (60 subjects)
Sleeplab AHI [15, 30]: 36 (36 subjects)
Sleeplab AHI [30, 100]: 18 (18 subjects)
Sleeplab AHI [15, 100]: 49 (49 subjects)


### create Xy_all, containing the ICU and the sleeplab strata of choice

In [13]:
vars_of_interest = ['rr_10s_smooth',  'ibi', 'instability_30sec', 'ventilation_cvar_30sec']
units = [' (cpm)', ' (s)','' , '']
titles = ['Resp. Rate', 'Inter-Breath-Interval', 'Variability Index', 'Ventilation CV']

stages = ['W', 'S', 'R', 'N1', 'N2', 'N3']
stages_titles = ['Wake', 'Sleep', 'R', 'N1', 'N2', 'N3']
mode = 'median'

Xy_all = pd.DataFrame() # columns=new_cols)

for i_stage, stage in enumerate(stages):
    
    Xy_stage = pd.DataFrame()
    
    vars_sel = [x + f'_stagewise_agrelaxed_ecg_nn_{stage}' for x in vars_of_interest]

    for var_tmp in vars_sel:
        sleeplab_all[var_tmp + '_iqr'] = sleeplab_all[var_tmp + '_q75'] - sleeplab_all[var_tmp + '_q25']
        sleeplab_ahi_0_5[var_tmp + '_iqr'] = sleeplab_ahi_0_5[var_tmp + '_q75'] - sleeplab_ahi_0_5[var_tmp + '_q25']
        sleeplab_ahi_15_100[var_tmp + '_iqr'] = sleeplab_ahi_15_100[var_tmp + '_q75'] - sleeplab_ahi_15_100[var_tmp + '_q25']
        summary_dn_icu_sel[var_tmp + '_iqr'] = summary_dn_icu_sel[var_tmp + '_q75'] -  summary_dn_icu_sel[var_tmp + '_q25']
        

    cols_sel = [x + '_std' for x in vars_sel] + [x + '_mean' for x in vars_sel] + [x + '_iqr' for x in vars_sel] + [x + '_median' for x in vars_sel] + \
    ['strata'] + ['strata_int'] + ['study_id']

    
    Xy_stage = pd.concat([sleeplab_ahi_15_100[cols_sel],
               sleeplab_ahi_0_5[cols_sel],
               summary_dn_icu_sel[cols_sel]],
              axis=0)
                
    if i_stage > 0:
        Xy_stage.drop(['strata', 'strata_int', 'study_id'], axis=1, inplace=True)
    
    Xy_all = pd.concat([Xy_all, Xy_stage], axis=1).copy()

In [27]:
Xy_all.shape

(281, 99)

In [14]:
Xy_all.strata.value_counts()

ICU           163
ahi_0_5        69
ahi_15_100     49
Name: strata, dtype: int64

In [14]:
Xy_all.strata.value_counts()

ICU           129
ahi_0_5        69
ahi_15_100     49
Name: strata, dtype: int64

### SUMMARY TABLES, including Statistical Tests

In [15]:
from scipy.stats import shapiro, mannwhitneyu, median_test, ttest_ind, normaltest, shapiro

#### first, compute statistics (mean, std, median, iqr) for both breathing and hrv features

In [16]:
# Mean HRV and Breathing.
variables_template = [f'rr_10s_smooth_stagewise_AGREEMENT_MODALITY_STAGE',
                      'ibi_stagewise_AGREEMENT_MODALITY_STAGE',
                      'instability_30sec_stagewise_AGREEMENT_MODALITY_STAGE',
                      'ventilation_cvar_30sec_stagewise_AGREEMENT_MODALITY_STAGE']

variables_template += [f'hr_stagewise_AGREEMENT_MODALITY_STAGE',
                       f'hrv_nnmean_stagewise_AGREEMENT_MODALITY_STAGE',
                      'hrv_rmssd_stagewise_AGREEMENT_MODALITY_STAGE',
                      'hrv_vlf_stagewise_AGREEMENT_MODALITY_STAGE',
                      'hrv_lf_stagewise_AGREEMENT_MODALITY_STAGE',
                     'hrv_hf_stagewise_AGREEMENT_MODALITY_STAGE']

# x_labels = ['Sleep Dur. (h)', 'Discordant S. (h)',  'Sleep (%)', 'Stage R (%)', 'Stage N1 (%)', 'Stage N2 (%)', 'Stage N3 (%)', 'Stage N2+N3(%)', 'SFI', 'Wake Trans./h']
x_labels_short = ['RR', 'IBI', 'Var', 'Vent']
stages = ['S', 'N1', 'N2', 'N3', 'R', 'W']


for stage in stages:
    
    modality = 'amendsumeffort'
    variables_breathing = [x.replace('MODALITY', modality).replace('AGREEMENT', agreement_short).replace('STAGE', stage) for x in variables_template]
    modality = 'ecg_nn'
    variables_hrv = [x.replace('MODALITY', modality).replace('AGREEMENT', agreement_short).replace('STAGE', stage) for x in variables_template]
    modality = 'mean'
    variables_mean = [x.replace('MODALITY', modality).replace('AGREEMENT', agreement_short).replace('STAGE', stage) for x in variables_template]

    for var_tmp in variables_breathing + variables_hrv:
        summary_dn_icu_sel[var_tmp + f'_iqr'] = summary_dn_icu_sel[var_tmp + f'_q75'] - summary_dn_icu_sel[var_tmp + f'_q25']
        sleeplab_all[var_tmp + f'_iqr'] = sleeplab_all[var_tmp + f'_q75'] - sleeplab_all[var_tmp + f'_q25']
        sleeplab_ahi_0_5[var_tmp + f'_iqr'] = sleeplab_ahi_0_5[var_tmp + f'_q75'] - sleeplab_ahi_0_5[var_tmp + f'_q25']
        sleeplab_ahi_15_100[var_tmp + f'_iqr'] = sleeplab_ahi_15_100[var_tmp + f'_q75'] - sleeplab_ahi_15_100[var_tmp + f'_q25']

    for statistic in ['mean', 'std', 'median', 'iqr']:
        for var_breathing, var_hrv, var_mean in zip(variables_breathing, variables_hrv, variables_mean):
            summary_dn_icu_sel[var_mean + f'_{statistic}'] = (summary_dn_icu_sel[var_breathing + f'_{statistic}'] + summary_dn_icu_sel[var_hrv+ f'_{statistic}']) / 2
            sleeplab_all[var_mean + f'_{statistic}'] = (sleeplab_all[var_breathing + f'_{statistic}'] + sleeplab_all[var_hrv+ f'_{statistic}']) / 2
            sleeplab_ahi_0_5[var_mean + f'_{statistic}'] = (sleeplab_ahi_0_5[var_breathing + f'_{statistic}'] + sleeplab_ahi_0_5[var_hrv+ f'_{statistic}']) / 2
            sleeplab_ahi_15_100[var_mean + f'_{statistic}'] = (sleeplab_ahi_15_100[var_breathing + f'_{statistic}'] + sleeplab_ahi_15_100[var_hrv+ f'_{statistic}']) / 2

In [29]:
print(sleeplab_all.shape[0])
print(sleeplab_ahi_0_5.shape[0])
print(sleeplab_ahi_15_100.shape[0])

190
69
49


#### compute one row for one ICU patient


In [17]:
summary_dn_icu_sel_subject = summary_dn_icu_sel.loc[:, summary_dn_icu_sel.dtypes != 'object'].copy()
for study_id_tmp in summary_dn_icu_sel.study_id.unique():
    
    new_values = pd.DataFrame(summary_dn_icu_sel_subject.loc[summary_dn_icu_sel_subject.study_id == study_id_tmp].mean()).transpose()
    summary_dn_icu_sel_subject = summary_dn_icu_sel_subject[summary_dn_icu_sel_subject.study_id != study_id_tmp]
    summary_dn_icu_sel_subject = pd.concat([summary_dn_icu_sel_subject, new_values], axis=0, ignore_index=True)

#### set up table

In [18]:
cohorts_names = ['ICU', 'Sleeplab All', 'Sleeplab AHI 0-5', 'Sleeplab AHI 15-100']

statistics = ['N_Subj', 'Mean', 'Median']
statistics_pop = list(itertools.product(cohorts_names, statistics))
statistics = [ 'p_tt', 's_tt', 'p_medians', 's_medians', 'p_mwu', 's_mwu', 'Std', 'Iqr', 'p_dag', 'p_sha', 'gaussian',]
statistics_pop += list(itertools.product(cohorts_names, statistics))

stagewise_features = list(itertools.product(stages, x_labels_short))

df_results = pd.DataFrame(index=stagewise_features, columns = statistics_pop)

#### compute table for Breathing

In [19]:
variables_template = [f'rr_10s_smooth_stagewise_AGREEMENT_MODALITY_STAGE',
                      'ibi_stagewise_AGREEMENT_MODALITY_STAGE',
                      'instability_30sec_stagewise_AGREEMENT_MODALITY_STAGE',
                      'ventilation_cvar_30sec_stagewise_AGREEMENT_MODALITY_STAGE']

alpha_normal = 0.05
alpha_diff = 0.05
modality = 'std'
icu_data = 'subjects'
statistics = ['mean', 'std', 'median', 'iqr']
for statistic in ['mean', 'std']: # e.g. computed mean _rr_10_s over a night for each patient.

    if icu_data == 'segments':
        icu_data_sel = summary_dn_icu_sel
    if icu_data == 'subjects':
        icu_data_sel = summary_dn_icu_sel_subject

    for stage in stages:

        # get stagewise variable names:
        modality = 'amendsumeffort'
        variables_breathing = [x.replace('MODALITY', modality).replace('AGREEMENT', agreement_short).replace('STAGE', stage) for x in variables_template]
        modality = 'ecg_nn'
        variables_hrv = [x.replace('MODALITY', modality).replace('AGREEMENT', agreement_short).replace('STAGE', stage) for x in variables_template]
        modality = 'mean'
        variables_mean = [x.replace('MODALITY', modality).replace('AGREEMENT', agreement_short).replace('STAGE', stage) for x in variables_template]

        if modality == 'mean':
            variables_sel = variables_mean
        elif modality == 'amendsumeffort':
            variables_sel = variables_breathing
        elif modality == 'ecg_nn':
            variables_sel = variables_hrv
        else:
            raise ValueError('Wrong modality name')

        # run test procedure:
        for i_var in range(len(variables_mean)):

            var_tmp = variables_sel[i_var] + f'_{statistic}'
            name_tmp = (stage, x_labels_short[i_var])

            icu_values = icu_data_sel[var_tmp].dropna()
            sl_all_values = sleeplab_all[var_tmp].dropna()
            sl_ahi_0_5_values = sleeplab_ahi_0_5[var_tmp].dropna()
            sl_ahi_15_100_values = sleeplab_ahi_15_100[var_tmp].dropna()

            cohorts_values = [icu_values, sl_all_values, sl_ahi_0_5_values, sl_ahi_15_100_values]

            for cohort_values, cohort_name in zip(cohorts_values, cohorts_names):

                if cohort_values.mean() < 1:
                    decimals = 2
                else: 
                    decimals = 1

                df_results.loc[name_tmp, (cohort_name, 'N_Subj')] = len(cohort_values)
                # get mean and std:
                df_results.loc[name_tmp, (cohort_name, 'Mean')] = cohort_values.mean().round(decimals)
                df_results.loc[name_tmp, (cohort_name, 'Std')] = cohort_values.std().round(decimals)
                df_results.loc[name_tmp, (cohort_name, 'Median')] = cohort_values.median().round(decimals)
                df_results.loc[name_tmp, (cohort_name, 'Iqr')] = (cohort_values.quantile(0.75) - cohort_values.quantile(0.25)).round(decimals)
                # normality checks:
                p_dag = normaltest(cohort_values)[1]
                p_sha = shapiro(cohort_values)[1]
                gaussian = np.any([p_dag > alpha_normal, p_sha > alpha_normal])
                df_results.loc[name_tmp, (cohort_name, 'p_dag')] = p_dag.round(4)
                df_results.loc[name_tmp, (cohort_name, 'p_sha')] = np.round(p_sha, 4)
                df_results.loc[name_tmp, (cohort_name, 'gaussian')] = gaussian

                if cohort_name == 'ICU':
                    gaussian_icu = gaussian
                    continue

                df_results.loc[name_tmp, (cohort_name, 'p_tt')] = ttest_ind(icu_values, cohort_values)[1].round(4)
#                 df_results.loc[name_tmp, (cohort_name, 's_tt')] = ttest_ind(icu_values, cohort_values)[0].round(4)
                df_results.loc[name_tmp, (cohort_name, 'p_medians')] = median_test(icu_values, cohort_values)[1].round(4)
#                 df_results.loc[name_tmp, (cohort_name, 's_medians')] = median_test(icu_values, cohort_values)[0].round(4)
                df_results.loc[name_tmp, (cohort_name, 'p_mwu')] = mannwhitneyu(icu_values, cohort_values)[1].round(4)
#                 df_results.loc[name_tmp, (cohort_name, 's_mwu')] = mannwhitneyu(icu_values, cohort_values)[0].round(4)

                if gaussian & gaussian_icu:
                    significance_001 = ttest_ind(icu_values, cohort_values)[1] <= 0.001
                    significance_1 = ttest_ind(icu_values, cohort_values)[1] <= 0.01
                    significance_5 = ttest_ind(icu_values, cohort_values)[1] <= 0.05
                    if significance_1:
                        df_results.loc[name_tmp, (cohort_name, 'Mean')] = '*** ' + str(cohort_values.mean().round(1))
                    elif significance_1:
                        df_results.loc[name_tmp, (cohort_name, 'Mean')] = '** ' + str(cohort_values.mean().round(1))
                    elif significance_5:
                        df_results.loc[name_tmp, (cohort_name, 'Mean')] = '* ' + str(cohort_values.mean().round(1))

                else:
                    significance_001 = (median_test(icu_values, cohort_values)[1] <= 0.001) & (mannwhitneyu(icu_values, cohort_values)[1] <= 0.001)
                    significance_1 = (median_test(icu_values, cohort_values)[1] <= 0.01) & (mannwhitneyu(icu_values, cohort_values)[1] <= 0.01)
                    significance_5 = (median_test(icu_values, cohort_values)[1] <= 0.05) & (mannwhitneyu(icu_values, cohort_values)[1] <= 0.05)
                    if significance_001:
                        df_results.loc[name_tmp, (cohort_name, 'Median')] = '*** ' + str(cohort_values.median().round(1))
                    elif significance_1:
                        df_results.loc[name_tmp, (cohort_name, 'Median')] = '** ' + str(cohort_values.median().round(1))
                    elif significance_5:
                        df_results.loc[name_tmp, (cohort_name, 'Median')] = '* ' + str(cohort_values.median().round(1))

    df_results.dropna(axis=1, how='all', inplace=True)
    df_results.transpose().to_csv(os.path.join(tables_savedir, f'results_breathing_analysis_{modality}_{icu_data}_{statistic}_minhours{str(min_hours_sleep_icu)}_inclusion_{inclusion_agreement}_breathingstatistics_{agreement}_{icu_day_v}_sleeplabagree_{do_agreement_for_sleeplab}.csv'))
    df_results.to_csv(os.path.join(tables_savedir, f'results_breathing_analysis_{modality}_{icu_data}_{statistic}_minhours{str(min_hours_sleep_icu)}_inclusion_{inclusion_agreement}_breathingstatistics_{agreement}_{icu_day_v}_sleeplabagree_{do_agreement_for_sleeplab}_T.csv'))
    
    if statistic == 'mean':
        df_results_mean = df_results.copy()
    elif statistic == 'std':
        df_results_std = df_results.copy()

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/scipy/stats/stats.py:1604: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=16
  "anyway, n=%i" % int(n))


#### compute table for HRV

In [20]:
if 0:
    
    variables_template = [f'hr_stagewise_AGREEMENT_MODALITY_STAGE',
                      f'hrv_nnmean_stagewise_AGREEMENT_MODALITY_STAGE',
                      'hrv_rmssd_stagewise_AGREEMENT_MODALITY_STAGE',
                      'hrv_vlf_stagewise_AGREEMENT_MODALITY_STAGE',
                      'hrv_lf_stagewise_AGREEMENT_MODALITY_STAGE',
                     'hrv_hf_stagewise_AGREEMENT_MODALITY_STAGE']
    x_labels_short = ['HR', 'NN', 'RMSSD', 'VLF', 'LF', 'HF']

    import itertools
    # statistics = ['Mean', 'Std', 'Median', 'Iqr', 'p_dag', 'p_sha', 'gaussian', 'p_tt', 'p_medians', 'p_mwu', 'diff_tt', 'sig_medians', 'sig_mwu']
    # cohorts_names = ['ICU', 'Sleeplab All', 'Sleeplab AHI 0-5', 'Sleeplab AHI 15-100']
    # statistics_pop = list(itertools.product(cohorts_names, statistics))

    cohorts_names = ['ICU', 'Sleeplab All', 'Sleeplab AHI 0-5', 'Sleeplab AHI 15-100']

    statistics = ['Mean', 'Median']
    statistics_pop = list(itertools.product(cohorts_names, statistics))
    statistics = [ 'p_tt', 'p_medians', 'p_mwu', 'Std',  'Iqr', 'p_dag', 'p_sha', 'gaussian',]
    statistics_pop += list(itertools.product(cohorts_names, statistics))

    stagewise_features = list(itertools.product(stages, x_labels_short))

    df_results = pd.DataFrame(index=stagewise_features, columns = statistics_pop)


    alpha_normal = 0.05
    alpha_diff = 0.05
    modality = 'mean'
    icu_data = 'subjects'
    statistics = ['mean', 'std', 'median', 'iqr']
    statistic = 'median' # e.g. computed mean _rr_10_s over a night for each patient.

    if icu_data == 'segments':
        icu_data_sel = summary_dn_icu_sel
    if icu_data == 'subjects':
        icu_data_sel = summary_dn_icu_sel_subject

    for stage in stages:

        # get stagewise variable names:
        modality = 'amendsumeffort'
        variables_breathing = [x.replace('MODALITY', modality).replace('AGREEMENT', agreement_short).replace('STAGE', stage) for x in variables_template]
        modality = 'ecg_nn'
        variables_hrv = [x.replace('MODALITY', modality).replace('AGREEMENT', agreement_short).replace('STAGE', stage) for x in variables_template]
        modality = 'mean'
        variables_mean = [x.replace('MODALITY', modality).replace('AGREEMENT', agreement_short).replace('STAGE', stage) for x in variables_template]

        if modality == 'mean':
            variables_sel = variables_mean
        elif modality == 'amendsumeffort':
            variables_sel = variables_breathing
        elif modality == 'ecg_nn':
            variables_sel = variables_hrv
        else:
            raise ValueError('Wrong modality name')

        # run test procedure:
        for i_var in range(len(variables_mean)):

            var_tmp = variables_sel[i_var] + f'_{statistic}'
            name_tmp = (stage, x_labels_short[i_var])

            icu_values = icu_data_sel[var_tmp].dropna()
            sl_all_values =  sleeplab_all[var_tmp].dropna()
            sl_ahi_0_5_values =  sleeplab_ahi_0_5[var_tmp].dropna()
            sl_ahi_15_100_values =  sleeplab_ahi_15_100[var_tmp].dropna()

            cohorts_values = [icu_values, sl_all_values, sl_ahi_0_5_values, sl_ahi_15_100_values]

            for cohort_values, cohort_name in zip(cohorts_values, cohorts_names):

                if cohort_values.mean() < 1:
                    decimals = 2
                else: 
                    decimals = 1

                # get mean and std:
                df_results.loc[name_tmp, (cohort_name, 'Mean')] = cohort_values.mean().round(decimals)
                df_results.loc[name_tmp, (cohort_name, 'Std')] = cohort_values.std().round(decimals)
                df_results.loc[name_tmp, (cohort_name, 'Median')] = cohort_values.median().round(decimals)
                df_results.loc[name_tmp, (cohort_name, 'Iqr')] = (cohort_values.quantile(0.75) - cohort_values.quantile(0.25)).round(decimals)
                # normality checks:
                p_dag = normaltest(cohort_values)[1]
                p_sha = shapiro(cohort_values)[1]
                gaussian = np.any([p_dag > alpha_normal, p_sha > alpha_normal])
                df_results.loc[name_tmp, (cohort_name, 'p_dag')] = p_dag.round(4)
                df_results.loc[name_tmp, (cohort_name, 'p_sha')] = np.round(p_sha, 4)
                df_results.loc[name_tmp, (cohort_name, 'gaussian')] = gaussian

                if cohort_name == 'ICU':
                    gaussian_icu = gaussian
                    continue

                df_results.loc[name_tmp, (cohort_name, 'p_tt')] = ttest_ind(icu_values, cohort_values)[1].round(4)
                df_results.loc[name_tmp, (cohort_name, 'p_medians')] = median_test(icu_values, cohort_values)[1].round(4)
                df_results.loc[name_tmp, (cohort_name, 'p_mwu')] = mannwhitneyu(icu_values, cohort_values)[1].round(4)

                if gaussian & gaussian_icu:
                    significance_001 = ttest_ind(icu_values, cohort_values)[1] <= 0.001
                    significance_1 = ttest_ind(icu_values, cohort_values)[1] <= 0.01
                    significance_5 = ttest_ind(icu_values, cohort_values)[1] <= 0.05
                    if significance_001:
                        print(name_tmp)
                        df_results.loc[name_tmp, (cohort_name, 'Mean')] = '*** ' + str(cohort_values.mean().round(1))
                    elif significance_1:
                        df_results.loc[name_tmp, (cohort_name, 'Mean')] = '** ' + str(cohort_values.mean().round(1))
                    elif significance_5:
                        df_results.loc[name_tmp, (cohort_name, 'Mean')] = '* ' + str(cohort_values.mean().round(1))

                else:
                    significance_001 = (median_test(icu_values, cohort_values)[1] <= 0.001) & (mannwhitneyu(icu_values, cohort_values)[1] <= 0.001)
                    significance_1 = (median_test(icu_values, cohort_values)[1] <= 0.01) & (mannwhitneyu(icu_values, cohort_values)[1] <= 0.01)
                    significance_5 = (median_test(icu_values, cohort_values)[1] <= 0.05) & (mannwhitneyu(icu_values, cohort_values)[1] <= 0.05)
                    if significance_001:
                        print(name_tmp)
                        df_results.loc[name_tmp, (cohort_name, 'Median')] = '*** ' + str(cohort_values.median().round(1))
                    elif significance_1:
                        print(name_tmp)
                        df_results.loc[name_tmp, (cohort_name, 'Median')] = '** ' + str(cohort_values.median().round(1))
                    elif significance_5:
                        df_results.loc[name_tmp, (cohort_name, 'Median')] = '* ' + str(cohort_values.median().round(1))

    df_results.dropna(axis=1, how='all', inplace=True)
    df_results.transpose().to_csv(os.path.join(tables_savedir, f'results_hrv_analysis_{modality}_{icu_data}_{statistic}.csv'))
    df_results.to_csv(os.path.join(tables_savedir, f'results_hrv_analysis_{modality}_{icu_data}_{statistic}_T.csv'))

In [21]:
# NOTE: We take ecg_nn based sleep stage information to locate the sleep-based breathing features. however, two of the ICU subjects 
# were included due to breathing sleep but no ecg sleep, i.e. they show NAN values here. e.g. 165 in this example:
if 0:
    summary_dn_icu_sel.loc[summary_dn_icu_sel.study_id == 165, ['ventilation_cvar_30sec_stagewise_agrelaxed_ecg_nn_S_mean', 'ventilation_cvar_30sec_stagewise_agrelaxed_ecg_nn_S_std', f'hours_sleep_amendsumeffort_{agreement}', f'hours_sleep_ecg_nn_{agreement}']]

### breathing summary figure:

In [25]:
import re

xticks = [
    [12, 20, 28],
    [2.5, 5],
    [0.2, 0.4],
    [0.25, 0.50]
]
yticks = [
    [2, 5, 8],
    [0.5, 1.5, 2.5],
    [0.05, 0.15, 0.25],
    [0.05, 0.15, 0.25, 0.35],
    [10, 15, 17, 19],
    [2, 4, 6],
    [0.1, 0.2, 0.3],
    [0.2, 0.4],
    [3, 4, 5],
    [1, 2],
    [0.1, 0.2],
    [0.1, 0.2],
]

# Significance from Median needs to be transfered to Mean for this one, as other AHI groups use Mean.
df_results_mean.loc[[('S', 'IBI')], [('Sleeplab All', 'Mean')]] = '** ' + str(df_results_mean.loc[[('S', 'IBI')], [('Sleeplab All', 'Mean')]].values[0][0])


vars_to_plot = ['rr_10s_smooth',  'ibi', 'instability_30sec', 'ventilation_cvar_30sec']
stage = 'S'
vars_to_plot = [x + f'_stagewise_agrelaxed_ecg_nn_{stage}' for x in vars_to_plot]

ahi_strata = ['ahi_0_5', 'ahi_15_100']
ahi_colors = ['goldenrod', 'red']

units = [' (cpm)', ' (s)','' , '']
titles = ['Resp. Rate', 'Inter-Breath-Interval', 'Variability Index', 'Ventilation CVar']

fig, ax = plt.subplots(3, 4, figsize = (7, 4), constrained_layout=True, gridspec_kw={'height_ratios': [2,1,1]})
ax = ax.flatten()
s = 12
mode = 'mean'

plot_type = 'scatter'
colors = ['green', 'darkorange', 'blue']
markers = ['o', 's', 'P'] 

for i_var, var_tmp in enumerate(vars_to_plot):
    
    if mode == 'median':
        xlabel = 'Median' + units[i_var]
        ylabel = 'IQR' + units[i_var]
        x = var_tmp + '_median'
        y = var_tmp + '_iqr'
    elif mode == 'mean':
        xlabel = 'Mean' + units[i_var]
        ylabel = 'Std' + units[i_var]
        x = var_tmp + '_mean'
        y = var_tmp + '_std' 
        
    title = titles[i_var]
    
    if i_var == 0:
        legend ='full'
        
        Xy_print_tmp = Xy_all.loc[:, [x, y, 'strata', 'study_id']].copy()
        print(Xy_print_tmp['study_id'].unique().shape)
        print(f"N ICU: {Xy_print_tmp[Xy_print_tmp.strata=='ICU'].study_id.unique().shape[0]}")
#         print(f"N Sleeplab: {Xy_print_tmp[Xy_print_tmp.strata!='ICU'].study_id.unique().shape[0]}")
        print(f"N AHI0-5: {Xy_print_tmp[Xy_print_tmp.strata=='ahi_0_5'].study_id.unique().shape[0]}")
        print(f"N AHI15-100: {Xy_print_tmp[Xy_print_tmp.strata=='ahi_15_100'].study_id.unique().shape[0]}")
        
        Xy_print_tmp = Xy_print_tmp.dropna() # drop NA values for selected stage (usually sleep)
#         print(Xy_print_tmp['study_id'].unique().shape)
#         print(f"N ICU: {Xy_print_tmp[Xy_print_tmp.strata=='ICU'].study_id.unique().shape[0]}")
#         print(f"N Sleeplab: {Xy_print_tmp[Xy_print_tmp.strata!='ICU'].study_id.unique().shape[0]}")
#         print(f"N AHI0-5: {Xy_print_tmp[Xy_print_tmp.strata=='ahi_0_5'].study_id.unique().shape[0]}")
#         print(f"N AHI15-100: {Xy_print_tmp[Xy_print_tmp.strata=='ahi_15_100'].study_id.unique().shape[0]}")
    else:
        legend=False
    
    if plot_type == 'kde':
        alpha = 0.4
        sns.kdeplot(
            data=Xy_all.loc[:, [x, y, 'strata']], x=x, y=y, hue='strata', ax=ax[i_var], bw_adjust=0.85, levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.75],
            palette=colors, alpha=alpha, legend=legend,
        )
    elif plot_type == 'scatter':
        alpha = 0.3
        sns.scatterplot(
            data=Xy_all.loc[:, [x, y, 'strata']], x=x, y=y, hue='strata', style='strata', ax=ax[i_var], s=s,
            palette=colors, alpha=alpha, legend=legend, markers=markers,
        )

    ax[i_var].set_xlabel(xlabel, labelpad=1)
    if i_var == 0:
        ax[i_var].set_ylabel('Std', labelpad=1)
    else:
        ax[i_var].set_ylabel('')

    
# ax[0].legend(['Sleeplab AHI < 5', 'Sleeplab AHI > 15', 'ICU'], bbox_transform = ax[0].transAxes, bbox_to_anchor = (0.4, 1.3), loc='center', frameon=False, ncol=2, fontsize=8)

ax_row1 = ax[:4].flatten()
for i in range(len(ax_row1)):
    c_axis = 'black'
    ticklength = 1
    lw = 0.3
    for spine in ['bottom', 'top', 'right', 'left']:
        ax[i].spines[spine].set_color(c_axis)
        ax[i].spines[spine].set_linewidth(lw)
    ax[i].set_xticks(xticks[i])
    ax[i].set_yticks(yticks[i])
    ax[i].tick_params(axis='x', colors=c_axis, length=ticklength, direction='in', pad=-12)
    ax[i].tick_params(axis='y', colors=c_axis, length=ticklength, direction='in', pad=1)
    ax[i].yaxis.label.set_color(c_axis)
    ax[i].xaxis.label.set_color(c_axis)
    ax[i].xaxis.tick_top()
    ax[i].xaxis.set_label_position('top')
    
for i, ax_i in enumerate(ax[4:].flatten()):
    c_axis = 'black'
    ticklength = 1
    lw = 0.3
    for spine in ['bottom', 'top', 'right', 'left']:
        ax_i.spines[spine].set_color(c_axis)
        ax_i.spines[spine].set_linewidth(lw)
#     ax[i].tick_params(axis='x', colors=c_axis, length=ticklength, direction='in', pad=-15)
    ax_i.tick_params(axis='y', colors=c_axis, length=ticklength, direction='in', pad=1)
#     ax[i].yaxis.label.set_color(c_axis)
#     ax[i].xaxis.label.set_color(c_axis)
#     ax[i].xaxis.tick_top()
#     ax[i].xaxis.set_label_position('top')
    ax_i.set_xticks([])
    ax_i.set_yticks(yticks[i+4])
#     ax_i.set_yticks([])
    
    
from matplotlib.lines import Line2D
# colors = ['blue', 'green', 'darkorange']
colors = ['blue', 'green', 'darkorange']
markers = ['P', 'o', 's'] 

lines = [Line2D([0], [0], color='white', marker=markers[i], markerfacecolor=colors[i], markersize=8, linewidth=2) for i in range(3)]
labels = ['ICU', 'Sleeplab AHI < 5', 'Sleeplab AHI > 15']

ax[0].legend(lines, labels, 
             frameon=False, markerscale=1, ncol=3, bbox_to_anchor=(1.1, 0.9, 0.5, 0.5), loc='center')

y_pos = 1.35
ax[0].text( 
0.5, y_pos, titles[0],
ha='center', va='top', fontdict = font_headers,
transform=ax[0].transAxes
)
ax[1].text( 
0.5, y_pos, titles[1],
ha='center', va='top', fontdict = font_headers,
transform=ax[1].transAxes
)
ax[2].text( 
0.5, y_pos, titles[2],
ha='center', va='top', fontdict = font_headers,
transform=ax[2].transAxes
)
ax[3].text( 
0.5, y_pos, titles[3],
ha='center', va='top', fontdict = font_headers,
transform=ax[3].transAxes
)


for i_row, statistic in enumerate(['mean', 'std']):
    for i_axis, feature in enumerate(['RR', 'IBI', 'Var', 'Vent']):

        if statistic == 'mean':
            df_results_tmp = df_results_mean
            if feature in ['Vent', 'Var']:
                summary_center = 'Median'
                summary_spread = 'Iqr'
            elif feature in ['IBI', 'RR']:
                summary_center = 'Mean'
                summary_spread = 'Std'
        elif statistic == 'std':
            df_results_tmp = df_results_std
            if feature in ['RR', 'IBI']:
                summary_center = 'Median'
                summary_spread = 'Iqr'
            elif feature in ['Var', 'Vent']:
                summary_center = 'Mean'
                summary_spread = 'Std'

        vals_and_symbols = df_results_tmp.loc[[('S', feature)], [('ICU', summary_center), ('Sleeplab All', summary_center), ('Sleeplab AHI 0-5', summary_center), ('Sleeplab AHI 15-100', summary_center)]].values[0]
        vals = [np.float(str(x).replace('*', '')) for x in vals_and_symbols]
        symbols = [re.search('\*+', str(x)) for x in vals_and_symbols]
        symbols = [x[0] if x is not None else 'n.s.' for x in symbols]
        yerr = df_results_tmp.loc[[('S', feature)], [('ICU', summary_spread), ('Sleeplab All', summary_spread), ('Sleeplab AHI 0-5', summary_spread), ('Sleeplab AHI 15-100', summary_spread)]].values[0]

        heights = vals
        bars = np.arange(4)
        axis_no = (i_row+1)*4 + i_axis
#         ax[axis_no].bar(bars, heights, align='center', color=['blue', 'crimson', 'green', 'darkorange'], yerr=yerr)
        ax[axis_no].bar(bars, heights, align='center', color=['blue', 'crimson', 'green', 'darkorange'], yerr=yerr)
        ax[axis_no].set_xticks(np.arange(4))
        ax[axis_no].set_xticklabels(['ICU', 'Sleeplab All', 'AHI<5', 'AHI>15'], fontsize=6, rotation=25)
        ymin, ymax = ax[axis_no].get_ylim()
        ax[axis_no].set_ylim(np.min(heights)/2, ymax*1.1)
        if symbols[1] != 'n.s.':
            barplot_annotate_brackets(0, 1, symbols[1], bars, heights, ax[axis_no], dh=0.03)
        if symbols[2] != 'n.s.':
            barplot_annotate_brackets(0, 2, symbols[2], bars, heights, ax[axis_no], dh=0.07)
        if symbols[3] != 'n.s.':
            barplot_annotate_brackets(0, 3, symbols[3], bars, heights, ax[axis_no], dh=0.11)

ax[4].set_ylabel('Mean', labelpad=0)
ax[8].set_ylabel('Std', labelpad=2)

plt.subplots_adjust(left=0.04, bottom=0.1, right=0.995, top=0.86, wspace=0.2, hspace=0)
# plt.tight_layout()

plt.savefig(os.path.join(plots_savedir, f'Figure_Breathing_4_Features_AHI_Strata_wBars_{plot_type}_minhours{str(min_hours_sleep_icu)}_agree_{agreement}_{icu_day_v}_sleeplabagree_{do_agreement_for_sleeplab}.jpg'), dpi=500)
plt.savefig(os.path.join(plots_savedir, f'Figure_Breathing_4_Features_AHI_Strata_wBars_{plot_type}_minhours{str(min_hours_sleep_icu)}_agree_{agreement}_{icu_day_v}_sleeplabagree_{do_agreement_for_sleeplab}.svg'))

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

(177,)
N ICU: 80
N Sleeplab: 118
N AHI0-5: 69
N AHI15-100: 49
(165,)
N ICU: 63
N Sleeplab: 117
N AHI0-5: 69
N AHI15-100: 48


/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:217: UserWarning: This figure was using constrained_layout==True, but that is incompatible with subplots_adjust and or tight_layout: setting constrained_layout==False. 


In [23]:
plots_savedir

'/scr/wolfgang/Dropbox (Partners HealthCare)/SleepInICUandSleeplabPaper/Plots/breathing'

In [24]:
df_results_tmp

,"(ICU, N_Subj)","(ICU, Mean)","(ICU, Median)","(Sleeplab All, N_Subj)","(Sleeplab All, Mean)","(Sleeplab All, Median)","(Sleeplab AHI 0-5, N_Subj)","(Sleeplab AHI 0-5, Mean)","(Sleeplab AHI 0-5, Median)","(Sleeplab AHI 15-100, N_Subj)","(Sleeplab AHI 15-100, Mean)","(Sleeplab AHI 15-100, Median)","(ICU, Std)","(ICU, Iqr)","(ICU, p_dag)","(ICU, p_sha)","(ICU, gaussian)","(Sleeplab All, p_tt)","(Sleeplab All, p_medians)","(Sleeplab All, p_mwu)","(Sleeplab All, Std)","(Sleeplab All, Iqr)","(Sleeplab All, p_dag)","(Sleeplab All, p_sha)","(Sleeplab All, gaussian)","(Sleeplab AHI 0-5, p_tt)","(Sleeplab AHI 0-5, p_medians)","(Sleeplab AHI 0-5, p_mwu)","(Sleeplab AHI 0-5, Std)","(Sleeplab AHI 0-5, Iqr)","(Sleeplab AHI 0-5, p_dag)","(Sleeplab AHI 0-5, p_sha)","(Sleeplab AHI 0-5, gaussian)","(Sleeplab AHI 15-100, p_tt)","(Sleeplab AHI 15-100, p_medians)","(Sleeplab AHI 15-100, p_mwu)","(Sleeplab AHI 15-100, Std)","(Sleeplab AHI 15-100, Iqr)","(Sleeplab AHI 15-100, p_dag)","(Sleeplab AHI 15-100, p_sha)","(Sleeplab AHI 15-100, gaussian)"
"(S, RR)",63,4,4,188,3.4,** 3.2,69,3.3,** 3.1,48,3.5,3.4,1.5,1.5,0,0,False,0.0001,0.0079,0.0003,1,1.2,0,0,False,0.001,0.0053,0.0013,1,1.6,0.1131,0.0397,True,0.033,0.1007,0.0139,0.9,1,0,0.0007,False
"(S, IBI)",63,1.3,1.2,188,1.5,1.4,69,1.3,1.2,48,1.8,*** 1.8,0.6,0.8,0.0273,0.0023,False,0.0302,0.1558,0.0044,0.5,0.7,0.0193,0.0037,False,0.9192,1,0.3993,0.5,0.6,0.0098,0.0004,False,0,0,0,0.5,0.5,0.2397,0.7,True
"(S, Var)",63,0.11,0.11,188,*** 0.1,0.14,69,* 0.1,0.13,48,*** 0.2,0.16,0.04,0.06,0.1376,0.115,True,0,0.0001,0,0.04,0.05,0.587,0.8932,True,0.029,0.0814,0.0114,0.04,0.05,0.6756,0.7254,True,0,0,0,0.03,0.04,0.4303,0.3246,True
"(S, Vent)",63,0.12,0.11,188,*** 0.2,0.19,69,*** 0.2,0.17,48,*** 0.2,0.22,0.05,0.07,0.0722,0.0169,True,0,0,0,0.07,0.09,0.1575,0.0819,True,0,0.0005,0,0.06,0.09,0.6252,0.6715,True,0,0,0,0.06,0.07,0.5511,0.6306,True
"(N1, RR)",56,4.2,4.1,185,3.5,** 3.4,66,3.4,** 3.3,48,3.5,* 3.5,1.6,1.6,0.0001,0.0006,False,0,0.0086,0.0004,0.8,1,0.0035,0.0058,False,0.0011,0.0064,0.0019,0.9,1.3,0.1767,0.0697,True,0.0081,0.0106,0.009,0.7,0.9,0.3801,0.2366,True
"(N1, IBI)",56,1.5,1.4,185,1.9,*** 1.9,66,1.7,* 1.7,48,*** 2.1,2.1,0.6,0.9,0.1337,0.0111,True,0,0.0002,0,0.5,0.7,0.0289,0.0061,False,0.0318,0.0182,0.0135,0.6,1,0.001,0.0267,False,0,0,0,0.4,0.5,0.7549,0.6892,True
"(N1, Var)",56,0.12,0.12,185,0.15,*** 0.2,66,*** 0.1,0.15,48,*** 0.2,0.16,0.04,0.04,0.537,0.4416,True,0,0,0,0.03,0.04,0.0142,0.0017,False,0.0005,0.0001,0.0002,0.04,0.05,0.1681,0.0358,True,0,0,0,0.03,0.03,0.4446,0.7113,True
"(N1, Vent)",56,0.13,0.13,185,*** 0.2,0.22,66,*** 0.2,0.22,48,*** 0.2,0.23,0.05,0.07,0.3223,0.0955,True,0,0,0,0.06,0.09,0.1152,0.1011,True,0,0,0,0.07,0.09,0.0473,0.0653,True,0,0,0,0.06,0.08,0.9961,0.9356,True
"(N2, RR)",38,3.3,3.2,167,3.2,3,64,3.1,3,37,3.2,3,1.6,2.4,0.0124,0.0062,False,0.5638,0.3513,0.4885,1,1.3,0,0,False,0.4605,0.3058,0.4463,1.1,1.4,0.0225,0.0279,False,0.8004,0.418,0.4473,1.1,0.7,0,0.0002,False
"(N2, IBI)",38,1.2,0.9,167,1.3,1.2,64,1.2,1,37,1.6,** 1.5,0.7,1,0.047,0.0002,False,0.175,0.0519,0.0098,0.5,0.6,0.0013,0.0001,False,0.9558,0.3058,0.1724,0.5,0.6,0.0007,0.0002,False,0.0059,0.0039,0.0007,0.5,0.6,0.6308,0.392,True


### BACKUP from previous versions of plots, subplots etc.

In [15]:
if 0:
    
    # Xy_icu and #Xy_sleeplab are temporary names that get passed to plotting functions
    Xy_sleeplab = sleeplab_all
    Xy_icu = summary_dn_icu_sel

    vars_to_plot = ['rr_10s_smooth',  'ibi', 'instability_30sec', 'ventilation_cvar_30sec']
    stage = 'S'
    vars_to_plot = [x + f'_stagewise_agrelaxed_ecg_nn_{stage}' for x in vars_to_plot]


    units = [' (cpm)', ' (s)','' , '']
    titles = ['Resp. Rate', 'Inter-Breath-Interval', 'Variability Index', 'Ventilation CV']

    fig, ax = plt.subplots(2, 2, figsize = (5,5))
    ax = ax.flatten()
    s = 5
    alpha = 0.4
    mode = 'mean'

    for i_var, var_tmp in enumerate(vars_to_plot):

        if mode == 'median':
            xlabel = 'Median' + units[i_var]
            ylabel = 'IQR' + units[i_var]
            x_var = var_tmp + '_median'
            y_var = var_tmp + '_iqr'
        elif mode == 'mean':
            xlabel = 'Mean' + units[i_var]
            ylabel = 'Std' + units[i_var]
            x_var = var_tmp + '_mean'
            y_var = var_tmp + '_std'        

        title = titles[i_var]

        default_scatter_ax(ax[i_var], x_var, y_var, title)

    ax[0].legend(['Sleeplab', 'ICU'], bbox_transform = ax[0].transAxes, bbox_to_anchor = (0.4, 1.2), loc='center', frameon=False, ncol=2, fontsize=8)

    ax = ax.flatten().flatten()
    for i in range(len(ax)):
        c_axis = 'black'
        ticklength = 1
        lw = 0.3
        for spine in ['bottom', 'top', 'right', 'left']:
            ax[i].spines[spine].set_color(c_axis)
            ax[i].spines[spine].set_linewidth(lw)
        ax[i].tick_params(axis='x', colors=c_axis, length=ticklength)
        ax[i].tick_params(axis='y', colors=c_axis, length=ticklength)
        ax[i].yaxis.label.set_color(c_axis)
        ax[i].xaxis.label.set_color(c_axis)

    plt.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=-0.1, hspace=-0.1)

    plt.tight_layout()

    plt.savefig(os.path.join(plots_savedir, f'Figure_Breathing_4_Features_{mode}_{stage}.jpg'), dpi=500)
    plt.savefig(os.path.join(plots_savedir, f'Figure_Breathing_4_Features_{mode}_{stage}.svg'))

In [16]:
if 0:

    vars_to_plot_1 = ['rr_10s_smooth',  'ibi', 'instability_30sec', 'ventilation_cvar_30sec']
    units = [' (cpm)', ' (s)','' , '']
    titles = ['Resp. Rate', 'Inter-Breath-Interval', 'Variability Index', 'Ventilation CV']

    mode = 'median'


    no_stages_strata = len(stages)

    fig, ax = plt.subplots(no_stages_strata, 4, figsize = (7.5, no_stages_strata*1.6), sharex='col', sharey='col')
    s = 5
    alpha = 0.4

    for i_stage, stage in enumerate(stages):

        vars_to_plot = [x + f'_stagewise_agrelaxed_ecg_nn_{stage}' for x in vars_to_plot_1]

        for var_tmp in vars_to_plot:
            Xy_sleeplab[var_tmp + '_iqr'] = Xy_sleeplab[var_tmp + '_q75'] - Xy_sleeplab[var_tmp + '_q25']
            Xy_icu[var_tmp + '_iqr'] = Xy_icu[var_tmp + '_q75'] -  Xy_icu[var_tmp + '_q25']

        for i_var, var_tmp in enumerate(vars_to_plot):
            if mode == 'median':
                xlabel = 'Median' + units[i_var]
                ylabel = 'IQR' + units[i_var]
                x_var = var_tmp + '_median'
                y_var = var_tmp + '_iqr'
            elif mode == 'mean':
                xlabel = 'Mean' + units[i_var]
                ylabel = 'Std' + units[i_var]
                x_var = var_tmp + '_mean'
                y_var = var_tmp + '_std'        

            title = None # titles[i_var]

            default_scatter_ax(ax[i_stage, i_var], x_var, y_var, title)

    ax[-1, 0].set_xticks([10, 15, 20, 25, 30])

    ax[0, 0].legend(['Sleeplab', 'ICU'], bbox_transform = ax[0, 0].transAxes, bbox_to_anchor = (0.4, 1.35), loc='center', frameon=False, ncol=2, fontsize=8)

    ax[0,0].text( 
    0.5, 1.2, titles[0],
    ha='center', va='top', fontdict = font_headers,
    transform=ax[0,0].transAxes
    )
    ax[0,1].text( 
    0.5, 1.2, titles[1],
    ha='center', va='top', fontdict = font_headers,
    transform=ax[0,1].transAxes
    )
    ax[0,2].text( 
    0.5, 1.2, titles[2],
    ha='center', va='top', fontdict = font_headers,
    transform=ax[0,2].transAxes
    )
    ax[0,3].text( 
    0.5, 1.2, titles[3],
    ha='center', va='top', fontdict = font_headers,
    transform=ax[0,3].transAxes
    )

    for i_stage, stage_name in enumerate(stages_titles):
        ax[i_stage, 0].text( 
        -0.4, 0.5, stage_name,
        ha='center', va='center', fontdict = font_headers, rotation='vertical',
        transform=ax[i_stage, 0].transAxes
        )


    ax = ax.flatten()


    for i in range(len(ax)):
        c_axis = 'black'
        ticklength = 1
        lw = 0.3
        for spine in ['bottom', 'top', 'right', 'left']:
            ax[i].spines[spine].set_color(c_axis)
            ax[i].spines[spine].set_linewidth(lw)
        ax[i].tick_params(axis='x', colors=c_axis, length=ticklength)
        ax[i].tick_params(axis='y', colors=c_axis, length=ticklength)
        ax[i].yaxis.label.set_color(c_axis)
        ax[i].xaxis.label.set_color(c_axis)

    plt.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0, hspace=-0.1)
    plt.tight_layout()


    plt.savefig(os.path.join(plots_savedir, f'Figure_Breathing_4_Features_{mode}_sleepstagestrata.jpg'), dpi=500)
    plt.savefig(os.path.join(plots_savedir, f'Figure_Breathing_4_Features_{mode}_sleepstagestrata.svg'))

####
2. make above plot with ahi strata. scatter and kde.
3. run ttests for all those statistics for icu vs sleeplab and sleeplab strata

In [17]:
if 0:
    
    vars_to_plot = ['rr_10s_smooth',  'ibi', 'instability_30sec', 'ventilation_cvar_30sec']
    stage = 'S'
    vars_to_plot = [x + f'_stagewise_agrelaxed_ecg_nn_{stage}' for x in vars_to_plot]

    ahi_strata = ['ahi_0_5', 'ahi_15_100']
    ahi_colors = ['goldenrod', 'red']


    units = [' (cpm)', ' (s)','' , '']
    titles = ['Resp. Rate', 'Inter-Breath-Interval', 'Variability Index', 'Ventilation CV']

    fig, ax = plt.subplots(2, 2, figsize = (5,5))
    ax = ax.flatten()
    s = 5
    alpha = 0.4
    mode = 'mean'

    for i_var, var_tmp in enumerate(vars_to_plot):

        if mode == 'median':
            xlabel = 'Median' + units[i_var]
            ylabel = 'IQR' + units[i_var]
            x_var = var_tmp + '_median'
            y_var = var_tmp + '_iqr'
        elif mode == 'mean':
            xlabel = 'Mean' + units[i_var]
            ylabel = 'Std' + units[i_var]
            x_var = var_tmp + '_mean'
            y_var = var_tmp + '_std'        

        title = titles[i_var]

        ax[i_var].scatter(sleeplab_ahi_0_5[x_var], sleeplab_ahi_0_5[y_var], c=ahi_colors[0], s=s, alpha=alpha)
        ax[i_var].scatter(sleeplab_ahi_15_100[x_var], sleeplab_ahi_15_100[y_var], c=ahi_colors[1], s=s, alpha=alpha)
        ax[i_var].scatter(summary_dn_icu_sel[x_var], summary_dn_icu_sel[y_var], c='navy', s=s, alpha=alpha-0.1)
        ax[i_var].set_xlabel(xlabel)
        ax[i_var].set_ylabel(ylabel)
    #     ax.legend(['Sleeplab', 'ICU'])
        ax[i_var].set_title(title)

    ax[0].legend(['Sleeplab AHI < 5', 'Sleeplab AHI > 15', 'ICU'], bbox_transform = ax[0].transAxes, bbox_to_anchor = (0.4, 1.3), loc='center', frameon=False, ncol=2, fontsize=8)

    ax = ax.flatten().flatten()
    for i in range(len(ax)):
        c_axis = 'black'
        ticklength = 1
        lw = 0.3
        for spine in ['bottom', 'top', 'right', 'left']:
            ax[i].spines[spine].set_color(c_axis)
            ax[i].spines[spine].set_linewidth(lw)
        ax[i].tick_params(axis='x', colors=c_axis, length=ticklength)
        ax[i].tick_params(axis='y', colors=c_axis, length=ticklength)
        ax[i].yaxis.label.set_color(c_axis)
        ax[i].xaxis.label.set_color(c_axis)

    plt.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=-0.1, hspace=-0.1)
    plt.tight_layout()

    plt.savefig(os.path.join(plots_savedir, f'Figure_Breathing_4_Features_AHI_Strata_{mode}_{stage}_minhours{str(min_hours_sleep_icu)}_agree_{agreement}_{icu_day_v}_sleeplabagree_{do_agreement_for_sleeplab}.jpg'), dpi=500)
    plt.savefig(os.path.join(plots_savedir, f'Figure_Breathing_4_Features_AHI_Strata_{mode}_{stage}_minhours{str(min_hours_sleep_icu)}_agree_{agreement}_{icu_day_v}_sleeplabagree_{do_agreement_for_sleeplab}.svg'))

In [18]:
if 0:
    
    vars_to_plot_1 = ['rr_10s_smooth',  'ibi', 'instability_30sec', 'ventilation_cvar_30sec']
    units = [' (cpm)', ' (s)','' , '']
    titles = ['Resp. Rate', 'Inter-Breath-Interval', 'Variability Index', 'Ventilation CV']

    stages = ['W', 'S', 'R', 'N1', 'N2', 'N3']
    stages_titles = ['Wake', 'Sleep', 'R', 'N1', 'N2', 'N3']
    mode = 'mean'


    no_stages_strata = len(stages)

    fig, ax = plt.subplots(no_stages_strata, 4, figsize = (7.5, no_stages_strata*1.6), sharex='col', sharey='col')
    s = 5
    alpha = 0.4

    for i_stage, stage in enumerate(stages):

        vars_to_plot = [x + f'_stagewise_agrelaxed_ecg_nn_{stage}' for x in vars_to_plot_1]

        for i_var, var_tmp in enumerate(vars_to_plot):
            if mode == 'median':
                xlabel = 'Median' + units[i_var]
                ylabel = 'IQR' + units[i_var]
                x_var = var_tmp + '_median'
                y_var = var_tmp + '_iqr'
            elif mode == 'mean':
                xlabel = 'Mean' + units[i_var]
                ylabel = 'Std' + units[i_var]
                x_var = var_tmp + '_mean'
                y_var = var_tmp + '_std'        

            title = None # titles[i_var]

            ax[i_stage, i_var].scatter(sleeplab_ahi_0_5[x_var], sleeplab_ahi_0_5[y_var], c=ahi_colors[0], s=s, alpha=alpha)
            ax[i_stage, i_var].scatter(sleeplab_ahi_15_100[x_var], sleeplab_ahi_15_100[y_var], c=ahi_colors[1], s=s, alpha=alpha)
            ax[i_stage, i_var].scatter(summary_dn_icu_sel[x_var], summary_dn_icu_sel[y_var], c='navy', s=s, alpha=alpha-0.1)
            ax[i_stage, i_var].set_xlabel(xlabel)
            ax[i_stage, i_var].set_ylabel(ylabel)
            #     ax.legend(['Sleeplab', 'ICU'])
            ax[i_stage, i_var].set_title(title)



    ax[-1, 0].set_xticks([10, 15, 20, 25, 30])

    ax[0, 0].legend(['Sleeplab AHI < 5', 'Sleeplab AHI > 15', 'ICU'], bbox_transform = ax[0, 0].transAxes, bbox_to_anchor = (0.4, 1.48), loc='center', frameon=False, ncol=1, fontsize=8)

    ax[0,0].text( 
    0.5, 1.2, titles[0],
    ha='center', va='top', fontdict = font_headers,
    transform=ax[0,0].transAxes
    )
    ax[0,1].text( 
    0.5, 1.2, titles[1],
    ha='center', va='top', fontdict = font_headers,
    transform=ax[0,1].transAxes
    )
    ax[0,2].text( 
    0.5, 1.2, titles[2],
    ha='center', va='top', fontdict = font_headers,
    transform=ax[0,2].transAxes
    )
    ax[0,3].text( 
    0.5, 1.2, titles[3],
    ha='center', va='top', fontdict = font_headers,
    transform=ax[0,3].transAxes
    )

    for i_stage, stage_name in enumerate(stages_titles):
        ax[i_stage, 0].text( 
        -0.4, 0.5, stage_name,
        ha='center', va='center', fontdict = font_headers, rotation='vertical',
        transform=ax[i_stage, 0].transAxes
        )


    ax = ax.flatten()


    for i in range(len(ax)):
        c_axis = 'black'
        ticklength = 1
        lw = 0.3
        for spine in ['bottom', 'top', 'right', 'left']:
            ax[i].spines[spine].set_color(c_axis)
            ax[i].spines[spine].set_linewidth(lw)
        ax[i].tick_params(axis='x', colors=c_axis, length=ticklength)
        ax[i].tick_params(axis='y', colors=c_axis, length=ticklength)
        ax[i].yaxis.label.set_color(c_axis)
        ax[i].xaxis.label.set_color(c_axis)

    plt.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0, hspace=-0.1)
    plt.tight_layout()


    # plt.savefig(os.path.join(plots_savedir, f'Figure_Breathing_4_Features_{mode}_sleepstagestrata.jpg'), dpi=500)
    # plt.savefig(os.path.join(plots_savedir, f'Figure_Breathing_4_Features_{mode}_sleepstagestrata.svg'))



In [19]:
print(f"N ICU: {Xy_all[Xy_all.strata=='ICU'].study_id.unique().shape[0]}")
print(f"N Sleeplab: {Xy_all[Xy_all.strata!='ICU'].study_id.unique().shape[0]}")
print(f"N AHI0-5: {Xy_all[Xy_all.strata=='ahi_0_5'].study_id.unique().shape[0]}")
print(f"N AHI15-100: {Xy_all[Xy_all.strata=='ahi_15_100'].study_id.unique().shape[0]}")

N ICU: 65
N Sleeplab: 118
N AHI0-5: 69
N AHI15-100: 49


In [21]:
if 0:
    
    vars_to_plot = ['rr_10s_smooth',  'ibi', 'instability_30sec', 'ventilation_cvar_30sec']
    stage = 'S'
    vars_to_plot = [x + f'_stagewise_agrelaxed_ecg_nn_{stage}' for x in vars_to_plot]

    ahi_strata = ['ahi_0_5', 'ahi_15_100']
    ahi_colors = ['goldenrod', 'red']

    units = [' (cpm)', ' (s)','' , '']
    titles = ['Resp. Rate', 'Inter-Breath-Interval', 'Variability Index', 'Ventilation CV']

    fig, ax = plt.subplots(2, 2, figsize = (3.5, 3.5), constrained_layout=True)
    ax = ax.flatten()
    s = 12
    mode = 'mean'

    plot_type = 'scatter'

    for i_var, var_tmp in enumerate(vars_to_plot):

        if mode == 'median':
            xlabel = 'Median' + units[i_var]
            ylabel = 'IQR' + units[i_var]
            x = var_tmp + '_median'
            y = var_tmp + '_iqr'
        elif mode == 'mean':
            xlabel = 'Mean' + units[i_var]
            ylabel = 'Std' + units[i_var]
            x = var_tmp + '_mean'
            y = var_tmp + '_std'        

        title = titles[i_var]

        if i_var == 0:
            legend ='full'
        else:
            legend=False

        if plot_type == 'kde':
            alpha = 0.4
            sns.kdeplot(
                data=Xy_all.loc[:, [x, y, 'strata']], x=x, y=y, hue='strata', ax=ax[i_var], bw_adjust=0.85, levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.75],
                palette=['green', 'darkorange', 'blue'], alpha=alpha, legend=legend,
            )
        elif plot_type == 'scatter':
            alpha = 0.3
            sns.scatterplot(
                data=Xy_all.loc[:, [x, y, 'strata']], x=x, y=y, hue='strata', ax=ax[i_var], s=s,
                palette=['green', 'darkorange', 'blue'], alpha=alpha, legend=legend,
            )

        ax[i_var].set_xlabel(xlabel, labelpad=0)
        ax[i_var].set_ylabel(ylabel, labelpad=0)


    # ax[0].legend(['Sleeplab AHI < 5', 'Sleeplab AHI > 15', 'ICU'], bbox_transform = ax[0].transAxes, bbox_to_anchor = (0.4, 1.3), loc='center', frameon=False, ncol=2, fontsize=8)

    ax = ax.flatten().flatten()
    for i in range(len(ax)):
        c_axis = 'black'
        ticklength = 1
        lw = 0.3
        for spine in ['bottom', 'top', 'right', 'left']:
            ax[i].spines[spine].set_color(c_axis)
            ax[i].spines[spine].set_linewidth(lw)
        ax[i].tick_params(axis='x', colors=c_axis, length=ticklength)
        ax[i].tick_params(axis='y', colors=c_axis, length=ticklength)
        ax[i].yaxis.label.set_color(c_axis)
        ax[i].xaxis.label.set_color(c_axis)

    from matplotlib.lines import Line2D
    colors = ['green', 'darkorange', 'blue']
    lines = [Line2D([0], [0], color=c, linewidth=2) for c in colors]
    labels = ['Sleeplab AHI < 5', 'Sleeplab AHI > 15', 'ICU']

    ax[0].legend(lines, labels, 
                 frameon=False, markerscale=1, ncol=2, bbox_to_anchor=(0.5, 0.95, 0.5, 0.5), loc='center')

    y_pos = 0.98
    ax[0].text( 
    0.5, y_pos, titles[0],
    ha='center', va='top', fontdict = font_headers,
    transform=ax[0].transAxes
    )
    ax[1].text( 
    0.5, y_pos, titles[1],
    ha='center', va='top', fontdict = font_headers,
    transform=ax[1].transAxes
    )
    ax[2].text( 
    0.5, y_pos, titles[2],
    ha='center', va='top', fontdict = font_headers,
    transform=ax[2].transAxes
    )
    ax[3].text( 
    0.5, y_pos, titles[3],
    ha='center', va='top', fontdict = font_headers,
    transform=ax[3].transAxes
    )

    plt.subplots_adjust(left=0.13, bottom=0.1, right=0.995, top=0.88, wspace=0.2, hspace=0.3)
    # plt.tight_layout()

    plt.savefig(os.path.join(plots_savedir, f'Figure_Breathing_4_Features_AHI_Strata_{plot_type}_minhours{str(min_hours_sleep_icu)}_agree_{agreement}_{icu_day_v}_sleeplabagree_{do_agreement_for_sleeplab}.jpg'), dpi=500)
    plt.savefig(os.path.join(plots_savedir, f'Figure_Breathing_4_Features_AHI_Strata_{plot_type}_minhours{str(min_hours_sleep_icu)}_agree_{agreement}_{icu_day_v}_sleeplabagree_{do_agreement_for_sleeplab}.svg'))

In [24]:
if 0:
    
    vars_to_plot_1 = ['rr_10s_smooth',  'ibi', 'instability_30sec', 'ventilation_cvar_30sec']
    units = [' (cpm)', ' (s)','' , '']
    titles = ['Resp. Rate', 'Inter-Breath-Interval', 'Variability Index', 'Ventilation CV']

    stages = ['W', 'S', 'R', 'N1', 'N2', 'N3']
    stages_titles = ['Wake', 'Sleep', 'R', 'N1', 'N2', 'N3']
    mode = 'mean'


    no_stages_strata = len(stages)

    fig, ax = plt.subplots(no_stages_strata, 4, figsize = (7.5, no_stages_strata*1.2), sharex='col', sharey='col', constrained_layout=True)
    # fig.subplots_adjust(hspace=0, wspace=0)
    s = 12
    plot_type = 'scatter'

    for i_stage, stage in enumerate(stages):

        vars_to_plot = [x + f'_stagewise_agrelaxed_ecg_nn_{stage}' for x in vars_to_plot_1]

    #     for var_tmp in vars_to_plot:
    #         Xy_sleeplab[var_tmp + '_iqr'] = Xy_sleeplab[var_tmp + '_q75'] - Xy_sleeplab[var_tmp + '_q25']
    #         Xy_icu[var_tmp + '_iqr'] = Xy_icu[var_tmp + '_q75'] -  Xy_icu[var_tmp + '_q25']

        for i_var, var_tmp in enumerate(vars_to_plot):
            if mode == 'median':
                xlabel = 'Median' + units[i_var]
                ylabel = 'IQR' + units[i_var]
                x_var = var_tmp + '_median'
                y_var = var_tmp + '_iqr'
            elif mode == 'mean':
                xlabel = 'Mean' + units[i_var]
                ylabel = 'Std' + units[i_var]
                x_var = var_tmp + '_mean'
                y_var = var_tmp + '_std'        

                if (i_stage==0) & (i_var == 0):
                    legend ='full'
                else:
                    legend=False

            title = None # titles[i_var]

            if plot_type == 'kde':
                alpha = 0.4
                sns.kdeplot(
                    data=Xy_all.loc[:, [x_var, y_var, 'strata']], x=x_var, y=y_var, hue='strata', ax=ax[i_stage, i_var], thresh=0.25, bw_adjust=0.85, levels=[0.1, 0.2, 0.5],
                    palette=['green', 'darkorange', 'blue'], alpha=alpha, legend=legend,
                )
            elif plot_type == 'scatter':
                alpha = 0.3
                sns.scatterplot(
                    data=Xy_all.loc[:, [x_var, y_var, 'strata']], x=x_var, y=y_var, hue='strata', ax=ax[i_stage, i_var], s=s,
                    palette=['green', 'darkorange', 'blue'], alpha=alpha, legend=legend,
                )

            ax[i_stage, i_var].set_xlabel(xlabel)
            ax[i_stage, i_var].set_ylabel(ylabel, labelpad=0)

    ax[-1, 0].set_xticks([10, 15, 20, 25, 30])

    # ax[0, 0].legend(['Sleeplab AHI < 5', 'Sleeplab AHI > 15', 'ICU'], bbox_transform = ax[0, 0].transAxes, bbox_to_anchor = (0.4, 0.5), 
    #                 loc='center', frameon=False, ncol=1, fontsize=8)

    # ax[0, 0].legend(bbox_to_anchor=(0, 0),borderaxespad=0, frameon=False)

    ax[0,0].text( 
    0.5, 1.2, titles[0],
    ha='center', va='top', fontdict = font_headers,
    transform=ax[0,0].transAxes
    )
    ax[0,1].text( 
    0.5, 1.2, titles[1],
    ha='center', va='top', fontdict = font_headers,
    transform=ax[0,1].transAxes
    )
    ax[0,2].text( 
    0.5, 1.2, titles[2],
    ha='center', va='top', fontdict = font_headers,
    transform=ax[0,2].transAxes
    )
    ax[0,3].text( 
    0.5, 1.2, titles[3],
    ha='center', va='top', fontdict = font_headers,
    transform=ax[0,3].transAxes
    )

    for i_stage, stage_name in enumerate(stages_titles):
        ax[i_stage, 0].text( 
        -0.4, 0.5, stage_name,
        ha='center', va='center', fontdict = font_headers, rotation='vertical',
        transform=ax[i_stage, 0].transAxes
        )



    ##########
    # legend_handles = []
    # figdummy, axdummy = plt.subplots(1,1);
    # axdummy.plot([0, 0], [1,2], color='green', zorder=1)
    # axdummy.plot([2, 2], [1,2], color='darkorange', zorder=1)
    # axdummy.plot([3, 3], [1,2], color='blue', zorder=1)
    # legend_handles.append(axdummy.plot([0, 0], [1,2], color='green', zorder=1))
    # legend_handles.append(axdummy.plot([2, 2], [1,2], color='darkorange', zorder=1))
    # legend_handles.append(axdummy.plot([3, 3], [1,2], color='blue', zorder=1))
    # axdummy.legend(legend_handles, ['Sleeplab AHI < 5', 'Sleeplab AHI > 15', 'ICU'], 
    #              frameon=False, markerscale=1, ncol=1, bbox_to_anchor=(0.2, 1, 0.5, 0.5), loc='best')

    ##########

    from matplotlib.lines import Line2D

    colors = ['green', 'darkorange', 'blue']
    lines = [Line2D([0], [0], color=c, linewidth=2) for c in colors]
    labels = ['Sleeplab AHI < 5', 'Sleeplab AHI > 15', 'ICU']

    ax[0, 0].legend(lines, labels, 
                 frameon=False, markerscale=1, ncol=2, bbox_to_anchor=(0.25, 1.2, 0.5, 0.5), loc='center')

    ##########

    ax = ax.flatten()

    for i in range(len(ax)):
        c_axis = 'black'
        ticklength = 1
        lw = 0.3
        for spine in ['bottom', 'top', 'right', 'left']:
            ax[i].spines[spine].set_color(c_axis)
            ax[i].spines[spine].set_linewidth(lw)
        ax[i].tick_params(axis='x', colors=c_axis, length=ticklength)
        ax[i].tick_params(axis='y', colors=c_axis, length=ticklength)
        ax[i].yaxis.label.set_color(c_axis)
        ax[i].xaxis.label.set_color(c_axis)
        if i > 0:
            ax[i].set_xticklabels([])


    plt.subplots_adjust(left=0.085, bottom=0.03, right=0.995, top=0.91, wspace=0.3, hspace=0)
    # plt.tight_layout()

    plt.savefig(os.path.join(plots_savedir, f'Figure_Breathing_4_Features_{mode}_{plot_type}_sleepstagestrata_minhours{str(min_hours_sleep_icu)}_agree_{agreement}_{icu_day_v}_sleeplabagree_{do_agreement_for_sleeplab}.jpg'), dpi=500)
    plt.savefig(os.path.join(plots_savedir, f'Figure_Breathing_4_Features_{mode}_{plot_type}_sleepstagestrata_minhours{str(min_hours_sleep_icu)}_agree_{agreement}_{icu_day_v}_sleeplabagree_{do_agreement_for_sleeplab}.svg'))

In [52]:
if 0:
    
    df_results_tmp = df_results_mean
    statistic = 'mean'
    for feature in ['RR']: #, 'IBI', 'Var', 'Vent']:

        if statistic == 'mean':
            if feature in ['Vent', 'Var']:
                summary_center = 'Median'
                summary_spread = 'Iqr'
            elif feature in ['IBI', 'RR']:
                summary_center = 'Mean'
                summary_spread = 'Std'
        elif statistic == 'std':
            if feature in ['RR', 'IBI']:
                summary_center = 'Median'
                summary_spread = 'Iqr'
            elif feature in ['Var', 'Vent']:
                summary_center = 'Mean'
                summary_spread = 'Std'

        vals_and_symbols = df_results_tmp.loc[[('S', feature)], [('ICU', summary_center), ('Sleeplab All', summary_center), ('Sleeplab AHI 0-5', summary_center), ('Sleeplab AHI 15-100', summary_center)]].values[0]
        vals = [np.float(str(x).replace('*', '')) for x in vals_and_symbols]
        symbols = [re.search('\*+', str(x)) for x in vals_and_symbols]
        symbols = [x[0] if x is not None else 'n.s.' for x in symbols]

        yerr = df_results_tmp.loc[[('S', feature)], [('ICU', summary_spread), ('Sleeplab All', summary_spread), ('Sleeplab AHI 0-5', summary_spread), ('Sleeplab AHI 15-100', summary_spread)]].values[0]

        heights = vals
        bars = np.arange(4)

        fig, ax = plt.subplots(1,1)
        ax.bar(bars, heights, align='center', color=['blue', 'crimson', 'green', 'darkorange'], yerr=yerr)
        ax.set_xticks(np.arange(4))
        ax.set_xticklabels(['ICU', 'Sleeplab All', 'AHI<5', 'AHI>15'])
        if symbols[1] != 'n.s.':
            barplot_annotate_brackets(0, 1, symbols[1], bars, heights, ax, dh=0.03)
        if symbols[2] != 'n.s.':
            barplot_annotate_brackets(0, 2, symbols[2], bars, heights, ax, dh=0.07)
        if symbols[3] != 'n.s.':
            barplot_annotate_brackets(0, 3, symbols[3], bars, heights, ax, dh=0.11)
        ax.set_ylim(np.min(heights)/2, None)


In [50]:
plt.close('all)')

### table with median breathing features (and also HRV) for each sleep stage and cohort strata, including statistical tests

#### breathing features

In [18]:
breathing_features = ['rr_10s_smooth',  'ibi', 'instability_30sec', 'ventilation_cvar_30sec']

    
import itertools
stages_vars = list(itertools.product(stages, [x_var, y_var], ['val', 'p_mwu', 'p_med']))
var_pop = list(itertools.product(titles, ['ICU', 'Sleeplab All', 'Sleeplab 0-5', 'Sleeplab 15-100']))
# var_pop = [x[0] + ' ' + x[1] for x in var_pop]

medians_df = pd.DataFrame(columns=stages_vars, index=var_pop)

In [19]:
from scipy.stats import shapiro, mannwhitneyu, median_test

In [20]:
for i_stage, stage in enumerate(stages):
    
    vars_to_plot = [x + f'_stagewise_agrelaxed_ecg_nn_{stage}' for x in breathing_features]

    for i_var, var_tmp in enumerate(vars_to_plot):
         
        x_sleeplab_all = sleeplab_all[var_tmp + '_' + x_var].dropna()
        medians_df.loc[(titles[i_var], 'Sleeplab All'), (stage, x_var, 'val')] = x_sleeplab_all.median().round(2)
        y_sleeplab_all = sleeplab_all[var_tmp + '_' + y_var].dropna()
        medians_df.loc[(titles[i_var], 'Sleeplab All'), (stage, y_var, 'val')] = y_sleeplab_all.median().round(2)
        x_sleeplab_0_5 = sleeplab_ahi_0_5[var_tmp + '_' + x_var].dropna()
        medians_df.loc[(titles[i_var], 'Sleeplab 0-5'), (stage, x_var, 'val')] = x_sleeplab_0_5.median().round(2)
        y_sleeplab_0_5 = sleeplab_ahi_0_5[var_tmp + '_' + y_var].dropna()
        medians_df.loc[(titles[i_var], 'Sleeplab 0-5'), (stage, y_var, 'val')] = y_sleeplab_0_5.median().round(2)
        x_sleeplab_15_100 = sleeplab_ahi_15_100[var_tmp + '_' + x_var].dropna()
        medians_df.loc[(titles[i_var], 'Sleeplab 15-100'), (stage, x_var, 'val')] = x_sleeplab_15_100.median().round(2)
        y_sleeplab_15_100 = sleeplab_ahi_15_100[var_tmp + '_' + y_var].dropna()
        medians_df.loc[(titles[i_var], 'Sleeplab 15-100'), (stage, y_var, 'val')] = y_sleeplab_15_100.median().round(2)
        x_icu = summary_dn_icu_sel[var_tmp + '_' + x_var].dropna()
        medians_df.loc[(titles[i_var], 'ICU'), (stage, x_var, 'val')] = x_icu.median().round(2)
        y_icu = summary_dn_icu_sel[var_tmp + '_' + y_var].dropna()
        medians_df.loc[(titles[i_var], 'ICU'), (stage, y_var, 'val')] = y_icu.median().round(2)
        
        medians_df.loc[(titles[i_var], 'Sleeplab All'), (stage, x_var, 'p_mwu')] = mannwhitneyu(x_sleeplab_all, x_icu)[1].round(4)
        medians_df.loc[(titles[i_var], 'Sleeplab All'), (stage, y_var, 'p_mwu')] = mannwhitneyu(y_sleeplab_all, y_icu)[1].round(4)
        medians_df.loc[(titles[i_var], 'Sleeplab 0-5'), (stage, x_var, 'p_mwu')] = mannwhitneyu(y_sleeplab_0_5, x_icu)[1].round(4)
        medians_df.loc[(titles[i_var], 'Sleeplab 0-5'), (stage, y_var, 'p_mwu')] = mannwhitneyu(y_sleeplab_0_5, y_icu)[1].round(4)
        medians_df.loc[(titles[i_var], 'Sleeplab 15-100'), (stage, x_var, 'p_mwu')] = mannwhitneyu(y_sleeplab_15_100, x_icu)[1].round(4)
        medians_df.loc[(titles[i_var], 'Sleeplab 15-100'), (stage, y_var, 'p_mwu')] = mannwhitneyu(y_sleeplab_15_100, y_icu)[1].round(4)
        
        medians_df.loc[(titles[i_var], 'Sleeplab All'), (stage, x_var, 'p_med')] = np.round(median_test(x_sleeplab_all, x_icu)[1], 4)
        medians_df.loc[(titles[i_var], 'Sleeplab All'), (stage, y_var, 'p_med')] = np.round(median_test(y_sleeplab_all, y_icu)[1], 4)
        medians_df.loc[(titles[i_var], 'Sleeplab 0-5'), (stage, x_var, 'p_med')] = np.round(median_test(y_sleeplab_0_5, x_icu)[1], 4)
        medians_df.loc[(titles[i_var], 'Sleeplab 0-5'), (stage, y_var, 'p_med')] = np.round(median_test(y_sleeplab_0_5, y_icu)[1], 4)
        medians_df.loc[(titles[i_var], 'Sleeplab 15-100'), (stage, x_var, 'p_med')] = np.round(median_test(y_sleeplab_15_100, x_icu)[1], 4)
        medians_df.loc[(titles[i_var], 'Sleeplab 15-100'), (stage, y_var, 'p_med')] = np.round(median_test(y_sleeplab_15_100, y_icu)[1], 4)   

In [21]:
median_iqr_results_breathing = medians_df.copy()
median_iqr_results_breathing.transpose().to_csv(f'median_iqr_results_breathing__minhours{str(min_hours_sleep_icu)}_agree_{agreement}_{icu_day_v}_sleeplabagree_{do_agreement_for_sleeplab}.csv')

#### HRV features

In [26]:
vars2 = ['hrv_nn',
         'hrv_ulf',
         'hrv_vlf',
         'hrv_lf',
         'hrv_hf',
         'hrv_lfhf',
         'hrv_nnmedian',
         'hrv_rmssd',
         'hrv_sd1',
         'hrv_sd2',
         'hrv_sd1sd2',
'cpc_hfc_lfc_ratio',
]

In [27]:
import itertools
stages_vars = list(itertools.product(stages, [x_var, y_var], ['val', 'p_mwu', 'p_med']))
var_pop = list(itertools.product(vars2, ['ICU', 'Sleeplab All', 'Sleeplab 0-5', 'Sleeplab 15-100']))
# var_pop = [x[0] + ' ' + x[1] for x in var_pop]

medians_df = pd.DataFrame(columns=stages_vars, index=var_pop)

In [28]:
for i_stage, stage in enumerate(stages):
    
    vars_to_plot = [x + f'_stagewise_agrelaxed_ecg_nn_{stage}' for x in vars2]

    for i_var, var_tmp in enumerate(vars_to_plot):
        
        sleeplab_all[var_tmp + '_iqr'] = sleeplab_all[var_tmp + '_q75'] - sleeplab_all[var_tmp + '_q25']
        sleeplab_ahi_0_5[var_tmp + '_iqr'] = sleeplab_ahi_0_5[var_tmp + '_q75'] - sleeplab_ahi_0_5[var_tmp + '_q25']
        sleeplab_ahi_15_100[var_tmp + '_iqr'] = sleeplab_ahi_15_100[var_tmp + '_q75'] - sleeplab_ahi_15_100[var_tmp + '_q25']
        summary_dn_icu_sel[var_tmp + '_iqr'] = summary_dn_icu_sel[var_tmp + '_q75'] - summary_dn_icu_sel[var_tmp + '_q25']

        x_sleeplab_all = sleeplab_all[var_tmp + '_' + x_var].dropna()
        medians_df.loc[(vars2[i_var], 'Sleeplab All'), (stage, x_var, 'val')] = x_sleeplab_all.median().round(2)
        y_sleeplab_all = sleeplab_all[var_tmp + '_' + y_var].dropna()
        medians_df.loc[(vars2[i_var], 'Sleeplab All'), (stage, y_var, 'val')] = y_sleeplab_all.median().round(2)
        x_sleeplab_0_5 = sleeplab_ahi_0_5[var_tmp + '_' + x_var].dropna()
        medians_df.loc[(vars2[i_var], 'Sleeplab 0-5'), (stage, x_var, 'val')] = x_sleeplab_0_5.median().round(2)
        y_sleeplab_0_5 = sleeplab_ahi_0_5[var_tmp + '_' + y_var].dropna()
        medians_df.loc[(vars2[i_var], 'Sleeplab 0-5'), (stage, y_var, 'val')] = y_sleeplab_0_5.median().round(2)
        x_sleeplab_15_100 = sleeplab_ahi_15_100[var_tmp + '_' + x_var].dropna()
        medians_df.loc[(vars2[i_var], 'Sleeplab 15-100'), (stage, x_var, 'val')] = x_sleeplab_15_100.median().round(2)
        y_sleeplab_15_100 = sleeplab_ahi_15_100[var_tmp + '_' + y_var].dropna()
        medians_df.loc[(vars2[i_var], 'Sleeplab 15-100'), (stage, y_var, 'val')] = y_sleeplab_15_100.median().round(2)
        x_icu = summary_dn_icu_sel[var_tmp + '_' + x_var].dropna()
        medians_df.loc[(vars2[i_var], 'ICU'), (stage, x_var, 'val')] = x_icu.median().round(2)
        y_icu = summary_dn_icu_sel[var_tmp + '_' + y_var].dropna()
        medians_df.loc[(vars2[i_var], 'ICU'), (stage, y_var, 'val')] = y_icu.median().round(2)
        
        medians_df.loc[(vars2[i_var], 'Sleeplab All'), (stage, x_var, 'p_mwu')] = mannwhitneyu(x_sleeplab_all, x_icu)[1].round(4)
        medians_df.loc[(vars2[i_var], 'Sleeplab All'), (stage, y_var, 'p_mwu')] = mannwhitneyu(y_sleeplab_all, y_icu)[1].round(4)
        medians_df.loc[(vars2[i_var], 'Sleeplab 0-5'), (stage, x_var, 'p_mwu')] = mannwhitneyu(y_sleeplab_0_5, x_icu)[1].round(4)
        medians_df.loc[(vars2[i_var], 'Sleeplab 0-5'), (stage, y_var, 'p_mwu')] = mannwhitneyu(y_sleeplab_0_5, y_icu)[1].round(4)
        medians_df.loc[(vars2[i_var], 'Sleeplab 15-100'), (stage, x_var, 'p_mwu')] = mannwhitneyu(y_sleeplab_15_100, x_icu)[1].round(4)
        medians_df.loc[(vars2[i_var], 'Sleeplab 15-100'), (stage, y_var, 'p_mwu')] = mannwhitneyu(y_sleeplab_15_100, y_icu)[1].round(4)
        
        medians_df.loc[(vars2[i_var], 'Sleeplab All'), (stage, x_var, 'p_med')] = np.round(median_test(x_sleeplab_all, x_icu)[1], 4)
        medians_df.loc[(vars2[i_var], 'Sleeplab All'), (stage, y_var, 'p_med')] = np.round(median_test(y_sleeplab_all, y_icu)[1], 4)
        medians_df.loc[(vars2[i_var], 'Sleeplab 0-5'), (stage, x_var, 'p_med')] = np.round(median_test(y_sleeplab_0_5, x_icu)[1], 4)
        medians_df.loc[(vars2[i_var], 'Sleeplab 0-5'), (stage, y_var, 'p_med')] = np.round(median_test(y_sleeplab_0_5, y_icu)[1], 4)
        medians_df.loc[(vars2[i_var], 'Sleeplab 15-100'), (stage, x_var, 'p_med')] = np.round(median_test(y_sleeplab_15_100, x_icu)[1], 4)
        medians_df.loc[(vars2[i_var], 'Sleeplab 15-100'), (stage, y_var, 'p_med')] = np.round(median_test(y_sleeplab_15_100, y_icu)[1], 4)    

In [29]:
median_iqr_results_hrv = medians_df.copy()
median_iqr_results_hrv.transpose().to_csv(f'median_iqr_results_hrv__minhours{str(min_hours_sleep_icu)}_agree_{agreement}_{icu_day_v}_sleeplabagree_{do_agreement_for_sleeplab}.csv')

In [32]:
median_iqr_results_hrv;